# Обучение DL-модели

В качетсве основгого фреймворка используем TenserFlow (Keras), для обучения, потому что:
- в библиотеке при обучении сразу считаются ключевые метрики, что упростит логирование
- проще экспериментировать с настройками моделей

## Устанавливаем и импортируем необходимые библиотеки

In [422]:
# !pip install clearml

In [423]:
# !pip install tensorflow

In [424]:
import numpy as np
import pandas as pd

import logging
from clearml import Task


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import mean_absolute_error

## Включаем логирование

По умолчанию будем использовать уровень INFO

In [425]:
logging.basicConfig(
    level = logging.INFO
)

**Подключим логирование в ClearML**

In [ ]:
NAME = 'FCN_BATCH_L2_REG_v17'

task_1 = Task.init(
    project_name = 'HSE-GP5',
    task_name = NAME,
    reuse_last_task_id = False
)

ClearML Task: created new task id=6d1f7c89e5024509b06f00543be0709a


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/e510d55020b94b9d8c50a57f8079f5b1/experiments/6d1f7c89e5024509b06f00543be0709a/output/log


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Зададим конфикурацию для экспериментов с FCN-моделями

In [427]:
config_to_experiments = {
    'activation': 'relu',
    'dropout': 0.3,
    'hidden_layers': [512, 256, 64],
    'learning_rate': 1e-3,
    'epochs': 50,
    'batch_size': 512,
    'embedding_dim': 32
}

Передаем конфигурацию в ClearML:

In [428]:
task_1.connect(config_to_experiments)

{'activation': 'relu',
 'dropout': 0.3,
 'hidden_layers': [512, 256, 64],
 'learning_rate': 0.001,
 'epochs': 50,
 'batch_size': 512,
 'embedding_dim': 32}

### Загрузка датасетов, полученных после EDA и предобработки

In [429]:
logging.info('....начало загрузки датасетов....')

INFO:root:....начало загрузки датасетов....


In [430]:
df_train = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_train.csv')
df_val = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_val.csv')
df_test = pd.read_csv('/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/datasets_final/df_test.csv')

In [431]:
logging.info('....загрузка датасетов закончена....')

INFO:root:....загрузка датасетов закончена....


**Определимся с назначением каждого из датасетов**

Предсказываем 'popularity': 0-100

In [432]:
all_columns = df_train.columns.to_list()
all_columns.remove('Unnamed: 0')
all_columns

['popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'artists_amount',
 'new_explicit',
 'track_genre_acoustic',
 'track_genre_afrobeat',
 'track_genre_alt-rock',
 'track_genre_alternative',
 'track_genre_ambient',
 'track_genre_anime',
 'track_genre_black-metal',
 'track_genre_bluegrass',
 'track_genre_blues',
 'track_genre_brazil',
 'track_genre_breakbeat',
 'track_genre_british',
 'track_genre_cantopop',
 'track_genre_chicago-house',
 'track_genre_children',
 'track_genre_chill',
 'track_genre_classical',
 'track_genre_club',
 'track_genre_comedy',
 'track_genre_country',
 'track_genre_dance',
 'track_genre_dancehall',
 'track_genre_death-metal',
 'track_genre_deep-house',
 'track_genre_detroit-techno',
 'track_genre_disco',
 'track_genre_disney',
 'track_genre_drum-and-bass',
 'track_genre_dub',
 'track_genre_dubstep',
 'track

In [433]:
df_train.drop(columns = 'Unnamed: 0', inplace = True)
df_val.drop(columns = 'Unnamed: 0', inplace = True)
df_test.drop(columns = 'Unnamed: 0', inplace = True)

In [434]:
PREDICT_VAR = 'popularity'
all_columns.remove('popularity')

ARTIST_NAME  = 'artist_label'
all_columns.remove('artist_label')

FEATURES_COLUMNS = all_columns

Теперь определение датасетов

In [435]:
X_train = df_train[FEATURES_COLUMNS].astype(float).values
X_val = df_val[FEATURES_COLUMNS].astype(float).values
X_test = df_test[FEATURES_COLUMNS].astype(float).values

In [436]:
Y_train = df_train[PREDICT_VAR].values
Y_val = df_val[PREDICT_VAR].values
Y_test = df_test[PREDICT_VAR].values

Отдельно обрабатываем столбец с именем артиста, поскольку ...

In [437]:
ART_train = df_train[ARTIST_NAME].values
ART_val = df_val[ARTIST_NAME].values
ART_test = df_test[ARTIST_NAME].values

## Строим DL-модель

Используем 2 входа Keras, поскольку в датасете есть столбцы закодированы 2 принципиально разными способома:
- OHE
- LabelEncoding

https://www.kaggle.com/code/hireme/two-inputs-neural-network-using-keras

In [438]:
numbers_input = keras.Input(shape = (X_train.shape[1],), name = 'numbers_feautures_input')

artists_input = keras.Input(shape = (1,), name = 'artists_feauture_input')

amount_of_artist_max = max(
    df_train['artist_label'].max(),
    df_val['artist_label'].max(),
    df_test['artist_label'].max()
) + 1

artist_embedding = layers.Embedding(
    input_dim = amount_of_artist_max,
    output_dim = config_to_experiments['embedding_dim'],
    name = 'artist_embedding'
)(artists_input)

artist_embedding = layers.Flatten()(artist_embedding)


x_full = layers.Concatenate()([numbers_input, artist_embedding])

**Создадим полносвязные слои для нейросети**

In [439]:
# for layer in config_to_experiments['hidden_layers']:
#     x_full = layers.Dense(
#         layer,
#         activation = config_to_experiments['activation']
#     )(x_full)

#     x_full = layers.Dropout(
#         config_to_experiments['dropout']
#     )(x_full)

# output_data = layers.Dense(1, name = 'output_data')(x_full)

Для L2 регуляризации вернемся к бейзлайну в конфигурации. Применим регуляризацию через настройки TenserFlow: https://www.tensorflow.org/api_docs/python/tf/keras/regularizers/L2

In [440]:
for layer in config_to_experiments['hidden_layers']:
    x_full = layers.Dense(
        layer,
        activation = config_to_experiments['activation'],
        kernel_regularizer = keras.regularizers.l2(1e-4)
    )(x_full)

    x_full = layers.Dropout(
        config_to_experiments['dropout']
    )(x_full)

output_data = layers.Dense(1, name = 'output_data')(x_full)

**Создадим сам объект модели**

In [441]:
model_task_1 = keras.Model(
    inputs = [numbers_input, artists_input],
    outputs = output_data
)

In [442]:
model_task_1.compile(
    optimizer = keras.optimizers.Adam(learning_rate = config_to_experiments['learning_rate']),
    loss = 'mse',
    metrics = ['mae']
)

Проверим, что все параметры корректны:

In [443]:
model_task_1.summary()

Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ artists_feauture_i… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ artist_embedding    │ (None, 1, 32)     │    486,912 │ artists_feauture… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numbers_feautures_… │ (None, 130)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_14          │ (None, 32)        │          0 │ artist_embedding… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_14      │ (None, 162)       │          0 │ numbers_feauture… │
│ (Concatenate)       │                   │            │ flatten_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_43 (Dense)    │ (None, 512)       │     83,456 │ concatenate_14[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_43          │ (None, 512)       │          0 │ dense_43[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 256)       │    131,328 │ dropout_43[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_44          │ (None, 256)       │          0 │ dense_44[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 64)        │     16,448 │ dropout_44[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_45          │ (None, 64)        │          0 │ dense_45[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_data (Dense) │ (None, 1)         │         65 │ dropout_45[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 718,209 (2.74 MB)

 Trainable params: 718,209 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

## Обучение модели

In [444]:
logging.info('Начало обучения модели')
logger = task_1.get_logger()

INFO:root:Начало обучения модели


In [445]:
learning_history = model_task_1.fit(
    [X_train, ART_train], y = Y_train,
    validation_data = ([X_val, ART_val], Y_val),
    epochs = config_to_experiments['epochs'],
    batch_size = config_to_experiments['batch_size'],
    verbose = 1
)

Epoch 1/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 477.2695 - mae: 16.4976 - val_loss: 224.4115 - val_mae: 10.3236
Epoch 2/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 212.4299 - mae: 10.2317 - val_loss: 206.6555 - val_mae: 9.5061
Epoch 3/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 186.4399 - mae: 9.3793 - val_loss: 203.9452 - val_mae: 9.2296
Epoch 4/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 175.1050 - mae: 8.9886 - val_loss: 202.9232 - val_mae: 9.1477
Epoch 5/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 170.2260 - mae: 8.8275 - val_loss: 202.2549 - val_mae: 9.0641
Epoch 6/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 167.4635 - mae: 8.7163 - val_loss: 201.5807 - val_mae: 9.0071
Epoch 7/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 165.0176 - mae: 8.6453 - val_loss: 203.8222 - val_mae: 9.0709
Epoch 8/50
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 162.8704 - mae: 8.5538 - val_loss: 203.0062 - val_mae: 9.0785
Epoch 9/50
123/123 ━

Сделаем логирование по эпохам обучения

In [446]:
for epoch, (train_mae_metric, val_mae_metric) in enumerate(zip(
    learning_history.history['mae'],
    learning_history.history['val_mae']
    )):

    logger.report_scalar('MAE', 'train', value = train_mae_metric, iteration = epoch)
    logger.report_scalar('MAE', 'val', value = val_mae_metric, iteration = epoch)

In [447]:
for epoch, (train_loss, val_loss) in enumerate(zip(
    learning_history.history['loss'],
    learning_history.history['val_loss']
    )):

    logger.report_scalar('MSE loss', 'train', value = train_loss, iteration = epoch)
    logger.report_scalar('MSE loss', 'val', value = val_loss, iteration = epoch)

In [448]:
logging.info('Проверка точности модели на test')

INFO:root:Проверка точности модели на test


In [449]:
y_pred = model_task_1.predict(
    [X_test, ART_test]
).flatten()

mae_mark_test = mean_absolute_error(Y_test, y_pred)
rmse_mark_test = np.sqrt(np.mean((Y_test - y_pred)**2))

421/421 ━━━━━━━━━━━━━━━━━━━━ 0s 726us/step


In [450]:
logging.info(f'\nTEST_MAE = {mae_mark_test}\nTEST_RMSE = {rmse_mark_test}')

logger.report_single_value(name = 'TEST_MAE', value = mae_mark_test)
logger.report_single_value(name = 'TEST_RMSE', value = rmse_mark_test)

INFO:root:
TEST_MAE = 8.556418418884277
TEST_RMSE = 14.238202558261342


## Сохранение моделей

In [451]:
PATH = f'/Users/anastasia/Desktop/HSE/HSE_3/СМАДИМО/GP_5/gp5/DL/FCN_models_task_1/{NAME}.keras'

model_task_1.save(PATH)

In [452]:
task_1.update_output_model(
    model_path = PATH,
    model_name = NAME
)

logging.info(f'Обучение модели {NAME} завершено успешно')
task_1.close()

INFO:root:Обучение модели FCN_BATCH_L2_REG_v17 завершено успешно
█████████████████████████████████ 100% | 8.27/8.27 MB [00:01<00:00,  8.13MB/s]: 
